In [1]:
import numpy as np
import pandas as pd

# Set random seed for reproducibility
np.random.seed(42)
num_records = 250

def generate_base_data(is_festival=False):
    # 1. Available charged batteries at 5:00 PM
    batteries_at_5pm = np.random.randint(10, 40, size=num_records)

    # 2. Batteries expected to finish charging by 7:00 PM
    batteries_charging_by_7pm = np.random.randint(5, 20, size=num_records)

    # Total supply available during the shift
    total_supply = batteries_at_5pm + batteries_charging_by_7pm

    # 3. Advance bookings
    advance_bookings = np.random.randint(5, 25, size=num_records)

    # 4. Historical walk-in demand (base)
    historical_walkin_demand = np.random.randint(10, 35, size=num_records)

    # 5. Rainfall forecast in mm (rain might decrease scooter usage/demand slightly or increase it for deliveries)
    rainfall_forecast_mm = np.round(np.random.exponential(scale=3.0, size=num_records), 1)

    # 6. Weekday/Weekend indicator (1 for weekday, 0 for weekend)
    is_weekday = np.random.choice([0, 1], size=num_records, p=[0.3, 0.7])

    # 7. Nearby-event indicator (1 for event, 0 for no event)
    nearby_event = np.random.choice([0, 1], size=num_records, p=[0.8, 0.2])

    # Simulate actual customer arrivals during the evening shift (5:00 PM - 7:00 PM)
    # Under a college festival (Version 2), walk-in demand and event factors are significantly boosted
    demand_multiplier = 1.5 if is_festival else 1.0

    # Calculate simulated walk-ins based on historical demand, adjusted by external factors
    simulated_walkins = []
    for i in range(num_records):
        base_walkin = historical_walkin_demand[i]
        # Adjustments
        event_boost = 12 if (nearby_event[i] == 1 or is_festival) else 0
        weekday_boost = 5 if is_weekday[i] == 1 else 0
        rain_reduction = -3 if rainfall_forecast_mm[i] > 5.0 else 0

        # Apply festival multiplier to the final walk-in simulation
        mean_demand = max(5, (base_walkin + event_boost + weekday_boost + rain_reduction) * demand_multiplier)
        actual_walkin = np.random.poisson(lam=mean_demand)
        simulated_walkins.append(actual_walkin)

    simulated_walkins = np.array(simulated_walkins)
    total_demand = advance_bookings + simulated_walkins

    # Target: 1 if demand exceeds supply, else 0
    battery_shortage = (total_demand > total_supply).astype(int)

    df = pd.DataFrame({
        'batteries_at_5pm': batteries_at_5pm,
        'batteries_charging_by_7pm': batteries_charging_by_7pm,
        'advance_bookings': advance_bookings,
        'historical_walkin_demand': historical_walkin_demand,
        'rainfall_forecast_mm': rainfall_forecast_mm,
        'is_weekday': is_weekday,
        'nearby_event': nearby_event,
        'simulated_total_demand': total_demand,
        'battery_shortage': battery_shortage
    })
    return df

# Create Version 1 (Standard Shift)
df_v1 = generate_base_data(is_festival=False)
df_v1.to_csv('scooter_batteries_v1_standard.csv', index=False)

# Create Version 2 (College Festival Period)
df_v2 = generate_base_data(is_festival=True)
df_v2.to_csv('scooter_batteries_v2_festival.csv', index=False)

print("--- VERSION 1: STANDARD SHIFT (First 5 records) ---")
print(df_v1.head())
print(f"Shortage Rate: {df_v1['battery_shortage'].mean() * 100:.1f}%")

print("\n--- VERSION 2: COLLEGE FESTIVAL PERIOD (First 5 records) ---")
print(df_v2.head())
print(f"Shortage Rate: {df_v2['battery_shortage'].mean() * 100:.1f}%")

--- VERSION 1: STANDARD SHIFT (First 5 records) ---
   batteries_at_5pm  batteries_charging_by_7pm  advance_bookings  \
0                16                          5                17   
1                29                          7                11   
2                38                         14                 8   
3                24                         16                17   
4                20                         12                24   

   historical_walkin_demand  rainfall_forecast_mm  is_weekday  nearby_event  \
0                        30                   2.7           0             0   
1                        31                   3.4           1             0   
2                        31                   4.7           0             1   
3                        12                   2.1           0             1   
4                        25                   0.3           0             0   

   simulated_total_demand  battery_shortage  
0                 

In [2]:
import pandas as pd
import numpy as np
import os

# 1. Simulate base aquaculture data for the pipeline
np.random.seed(42)
n_samples = 250

aquaculture_base = pd.DataFrame({
    'pond_id': np.random.randint(1, 10, size=n_samples),
    'feed_given_kg': np.random.uniform(50, 300, size=n_samples),
    'shrimp_biomass_kg': np.random.uniform(1000, 8000, size=n_samples),
    'aeration_run_hours': np.random.uniform(4, 24, size=n_samples),
    'pond_area_sq_meters': np.random.choice([1000, 2000, 5000], size=n_samples),
    'water_temp_celsius': np.random.uniform(25, 32, size=n_samples),
    'dissolved_oxygen_ppm': np.random.uniform(3.5, 7.5, size=n_samples),
    'harvest_survival_rate': np.random.uniform(0.6, 0.95, size=n_samples) # target proxy
})

# 2. Define the pipeline & feature engineering function
def run_aquaculture_pipeline(df):
    # Prevent modifications to original dataframe
    processed_df = df.copy()

    # Engineer required feature: feed_per_unit_biomass
    processed_df['feed_per_unit_biomass'] = processed_df['feed_given_kg'] / processed_df['shrimp_biomass_kg']

    # Engineer required feature: aeration_hours_per_area
    processed_df['aeration_hours_per_area'] = processed_df['aeration_run_hours'] / processed_df['pond_area_sq_meters']

    # Fill any potential division/inf errors
    processed_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    processed_df.fillna(0, inplace=True)

    return processed_df

# Execute Pipeline
engineered_dataset = run_aquaculture_pipeline(aquaculture_base)

# 3. Store the engineered features in a reusable feature file (Feature Store simulation)
feature_store_path = 'aquaculture_feature_store.csv'
engineered_dataset.to_csv(feature_store_path, index=False)
print(f"Success: Features stored successfully in '{feature_store_path}'\n")

# 4. Demonstrate loading features in a separate training step
print("--- Separate Training Step: Loading Features ---")
training_data = pd.read_csv(feature_store_path)

# Select the engineered features and potential target
features_to_use = ['feed_per_unit_biomass', 'aeration_hours_per_area', 'water_temp_celsius', 'dissolved_oxygen_ppm']
target = 'harvest_survival_rate'

X = training_data[features_to_use]
y = training_data[target]

print("Loaded Features Shape:", X.shape)
print("\nFirst 5 rows of engineered features:")
print(X.head())

Success: Features stored successfully in 'aquaculture_feature_store.csv'

--- Separate Training Step: Loading Features ---
Loaded Features Shape: (250, 4)

First 5 rows of engineered features:
   feed_per_unit_biomass  aeration_hours_per_area  water_temp_celsius  \
0               0.082805                 0.001297           29.586694   
1               0.033878                 0.009309           28.856002   
2               0.197592                 0.011383           25.608907   
3               0.024660                 0.001525           27.859172   
4               0.056384                 0.001066           27.608820   

   dissolved_oxygen_ppm  
0              4.488924  
1              6.983000  
2              5.281221  
3              5.559269  
4              4.936933  


In [5]:
!pip install -q mlflow

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
import mlflow
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')

# 1. Prepare Data
# Define "high-risk" ponds as those with a survival rate below 75% (0.75)
training_data = pd.read_csv('aquaculture_feature_store.csv')
training_data['high_risk_pond'] = (training_data['harvest_survival_rate'] < 0.75).astype(int)

features_to_use = ['feed_per_unit_biomass', 'aeration_hours_per_area', 'water_temp_celsius', 'dissolved_oxygen_ppm']
X = training_data[features_to_use]
y = training_data['high_risk_pond']

# Split into Train and Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Set up MLflow tracking
mlflow.set_experiment("Aquaculture_Risk_Classification")

# --- Model 1: Logistic Regression ---
with mlflow.start_run(run_name="Logistic_Regression") as lr_run:
    lr_model = LogisticRegression(random_state=42)
    lr_model.fit(X_train_scaled, y_train)
    lr_preds = lr_model.predict(X_test_scaled)

    # Evaluate
    lr_report = classification_report(y_test, lr_preds, output_dict=True)
    lr_f1 = f1_score(y_test, lr_preds)

    # Log to MLflow
    mlflow.log_params(lr_model.get_params())
    mlflow.log_metric("f1_score_high_risk", lr_report['1']['f1-score'])
    mlflow.log_metric("precision_high_risk", lr_report['1']['precision'])
    mlflow.log_metric("recall_high_risk", lr_report['1']['recall'])
    mlflow.sklearn.log_model(lr_model, "model")

# --- Model 2: Random Forest ---
with mlflow.start_run(run_name="Random_Forest") as rf_run:
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    rf_preds = rf_model.predict(X_test)

    # Evaluate
    rf_report = classification_report(y_test, rf_preds, output_dict=True)
    rf_f1 = f1_score(y_test, rf_preds)

    # Log to MLflow
    mlflow.log_params(rf_model.get_params())
    mlflow.log_metric("f1_score_high_risk", rf_report['1']['f1-score'])
    mlflow.log_metric("precision_high_risk", rf_report['1']['precision'])
    mlflow.log_metric("recall_high_risk", rf_report['1']['recall'])
    # Trusting the Tree structure to bypass skops UntrustedTypes Exception
    mlflow.sklearn.log_model(rf_model, "model", skops_trusted_types=["sklearn.tree._tree.Tree"])

# 3. Print Evaluation Comparison
print("=== LOGISTIC REGRESSION REPORT ===")
print(classification_report(y_test, lr_preds))

print("=== RANDOM FOREST REPORT ===")
print(classification_report(y_test, rf_preds))

# 4. Determine & Register Selected Best Model
best_model_name = "Aquaculture_Risk_Classifier_Best"
if rf_report['1']['f1-score'] > lr_report['1']['f1-score']:
    best_model = rf_model
    selected_run_id = rf_run.info.run_id
    print("Selected Model: Random Forest")
else:
    best_model = lr_model
    selected_run_id = lr_run.info.run_id
    print("Selected Model: Logistic Regression")

# Register the best model using the MLflow model registry
model_uri = f"runs:/{selected_run_id}/model"
mlflow.register_model(model_uri, best_model_name)
print(f"\nSuccess: Best model registered successfully in MLflow Registry under name: '{best_model_name}'")

2026/11/18 08:44:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/18 08:44:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'Aquaculture_Risk_Classifier_Best'.
2026/11/18 08:44:39 WARNING mlflow.tracking._model_registry.fluent: Run with id 9e66d882c64a4518848118471e15f380 has no artifacts at artifact path 'model', registering model based on models:/m-a0a8ed7b6f1f4d8ebda4c62e687d0c80 instead


=== LOGISTIC REGRESSION REPORT ===
              precision    recall  f1-score   support

           0       0.56      0.93      0.70        29
           1       0.00      0.00      0.00        21

    accuracy                           0.54        50
   macro avg       0.28      0.47      0.35        50
weighted avg       0.33      0.54      0.41        50

=== RANDOM FOREST REPORT ===
              precision    recall  f1-score   support

           0       0.61      0.76      0.68        29
           1       0.50      0.33      0.40        21

    accuracy                           0.58        50
   macro avg       0.56      0.55      0.54        50
weighted avg       0.56      0.58      0.56        50

Selected Model: Random Forest

Success: Best model registered successfully in MLflow Registry under name: 'Aquaculture_Risk_Classifier_Best'


Created version '1' of model 'Aquaculture_Risk_Classifier_Best'.


In [6]:
!pip install fastapi uvicorn mlflow scikit-learn pandas numpy joblib nest-asyncio

In [7]:
import mlflow
import mlflow.sklearn
import joblib
import os

MODEL_NAME = "Aquaculture_Risk_Classifier_Best"

# Load registered model from MLflow Registry
model = mlflow.sklearn.load_model(
    f"models:/{MODEL_NAME}/1"
)

# Save model locally for API serving
joblib.dump(model, "aquaculture_model.pkl")

# Save feature configuration
feature_columns = [
    "feed_per_unit_biomass",
    "aeration_hours_per_area",
    "water_temp_celsius",
    "dissolved_oxygen_ppm"
]

joblib.dump(feature_columns, "feature_columns.pkl")

print("Model loaded and saved successfully!")
print("Model:", MODEL_NAME)
print("Features:", feature_columns)

Model loaded and saved successfully!
Model: Aquaculture_Risk_Classifier_Best
Features: ['feed_per_unit_biomass', 'aeration_hours_per_area', 'water_temp_celsius', 'dissolved_oxygen_ppm']


In [9]:
%%writefile main.py

from fastapi import FastAPI
from pydantic import BaseModel, Field
import joblib
import pandas as pd

app = FastAPI(
    title="Aquaculture Oxygen Risk Prediction API",
    description="Predicts high-risk aquaculture ponds and recommends night-time aeration.",
    version="1.0"
)

# Load trained model and feature configuration
model = joblib.load("aquaculture_model.pkl")
feature_columns = joblib.load("feature_columns.pkl")


class PondInput(BaseModel):

    feed_given_kg: float = Field(gt=0)
    shrimp_biomass_kg: float = Field(gt=0)
    aeration_run_hours: float = Field(ge=0, le=24)
    pond_area_sq_meters: float = Field(gt=0)
    water_temp_celsius: float
    dissolved_oxygen_ppm: float = Field(ge=0)


@app.get("/health")
def health_check():

    return {
        "status": "healthy",
        "service": "Aquaculture Oxygen Risk Prediction API"
    }


@app.post("/predict")
def predict_oxygen_risk(data: PondInput):

    # Feature engineering
    feed_per_unit_biomass = (
        data.feed_given_kg / data.shrimp_biomass_kg
    )

    aeration_hours_per_area = (
        data.aeration_run_hours / data.pond_area_sq_meters
    )

    # Prepare model input
    input_data = pd.DataFrame([{
        "feed_per_unit_biomass": feed_per_unit_biomass,
        "aeration_hours_per_area": aeration_hours_per_area,
        "water_temp_celsius": data.water_temp_celsius,
        "dissolved_oxygen_ppm": data.dissolved_oxygen_ppm
    }])

    input_data = input_data[feature_columns]

    # Prediction
    prediction = int(model.predict(input_data)[0])

    # Probability of high risk
    probability = float(model.predict_proba(input_data)[0][1])

    oxygen_risk = "High" if prediction == 1 else "Normal"

    # Recommendation
    additional_aeration_required = oxygen_risk == "High"

    recommendation = (
        "Increase or extend night-time aeration and monitor oxygen levels."
        if additional_aeration_required
        else "Continue scheduled aeration and routine oxygen monitoring."
    )

    return {
        "oxygen_risk": oxygen_risk,
        "high_risk_probability": round(probability, 4),
        "additional_night_aeration_required": additional_aeration_required,
        "recommendation": recommendation
    }

Overwriting main.py


In [11]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run_api():
    uvicorn.run(
        "main:app",
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

INFO:     Started server process [516]
INFO:     Waiting for application startup.


In [12]:
import requests

response = requests.get("http://127.0.0.1:8000/health")

print("Status Code:", response.status_code)
print("Response:", response.json())

INFO:     127.0.0.1:37568 - "GET /health HTTP/1.1" 200 OK
Status Code: 200
Response: {'status': 'healthy', 'service': 'Aquaculture Oxygen Risk Prediction API'}


In [13]:
import requests

sample_pond = {
    "feed_given_kg": 200,
    "shrimp_biomass_kg": 5000,
    "aeration_run_hours": 8,
    "pond_area_sq_meters": 2000,
    "water_temp_celsius": 30,
    "dissolved_oxygen_ppm": 4.2
}

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=sample_pond
)

print("Status Code:", response.status_code)
print("Prediction:")
print(response.json())

INFO:     127.0.0.1:35480 - "POST /predict HTTP/1.1" 200 OK
Status Code: 200
Prediction:
{'oxygen_risk': 'Normal', 'high_risk_probability': 0.39, 'additional_night_aeration_required': False, 'recommendation': 'Continue scheduled aeration and routine oxygen monitoring.'}


In [14]:
invalid_pond = {
    "feed_given_kg": 200,
    "shrimp_biomass_kg": -500,
    "aeration_run_hours": 8,
    "pond_area_sq_meters": 2000,
    "water_temp_celsius": 30,
    "dissolved_oxygen_ppm": 4.2
}

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=invalid_pond
)

print("Status Code:", response.status_code)
print("Response:", response.json())

INFO:     127.0.0.1:46190 - "POST /predict HTTP/1.1" 422 Unprocessable Content
Status Code: 422
Response: {'detail': [{'type': 'greater_than', 'loc': ['body', 'shrimp_biomass_kg'], 'msg': 'Input should be greater than 0', 'input': -500, 'ctx': {'gt': 0.0}}]}


In [15]:
%%writefile test_api.py

from fastapi.testclient import TestClient
from main import app

client = TestClient(app)


def test_health():
    response = client.get("/health")

    assert response.status_code == 200
    assert response.json()["status"] == "healthy"


def test_prediction():
    data = {
        "feed_given_kg": 200,
        "shrimp_biomass_kg": 5000,
        "aeration_run_hours": 8,
        "pond_area_sq_meters": 2000,
        "water_temp_celsius": 30,
        "dissolved_oxygen_ppm": 4.2
    }

    response = client.post("/predict", json=data)

    assert response.status_code == 200

    result = response.json()

    assert result["oxygen_risk"] in ["High", "Normal"]
    assert 0 <= result["high_risk_probability"] <= 1
    assert isinstance(
        result["additional_night_aeration_required"], bool
    )


def test_invalid_input():
    data = {
        "feed_given_kg": 200,
        "shrimp_biomass_kg": -500,
        "aeration_run_hours": 8,
        "pond_area_sq_meters": 2000,
        "water_temp_celsius": 30,
        "dissolved_oxygen_ppm": 4.2
    }

    response = client.post("/predict", json=data)

    assert response.status_code == 422

Writing test_api.py


In [16]:
!pip install -q pytest httpx

In [17]:
!pytest test_api.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: platformdirs-4.14.3, langsmith-0.14.17, anyio-4.16.0, typeguard-4.6.0
collected 3 items                                                              

test_api.py::test_health PASSED                                          [ 33%]
test_api.py::test_prediction PASSED                                      [ 66%]
test_api.py::test_invalid_input PASSED                                   [100%]

============================== 3 passed in 3.20s ===============================


In [18]:
%%writefile requirements.txt

fastapi
uvicorn
pandas
numpy
scikit-learn
joblib
mlflow
pytest
httpx

Writing requirements.txt


In [19]:
import os

files_to_check = [
    "aquaculture_model.pkl",
    "feature_columns.pkl",
    "main.py",
    "test_api.py",
    "requirements.txt"
]

for file in files_to_check:
    print(f"{file}: {'EXISTS' if os.path.exists(file) else 'MISSING'}")

aquaculture_model.pkl: EXISTS
feature_columns.pkl: EXISTS
main.py: EXISTS
test_api.py: EXISTS
requirements.txt: EXISTS


In [20]:
!mkdir -p .github/workflows

In [21]:
%%writefile .github/workflows/ci.yml

name: Aquaculture Oxygen Risk CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: Validate model files
        run: |
          test -f aquaculture_model.pkl
          test -f feature_columns.pkl
          echo "Model files validated successfully"

      - name: Run automated tests
        run: pytest test_api.py -v

Writing .github/workflows/ci.yml


In [22]:
import os

files_to_check = [
    "main.py",
    "test_api.py",
    "requirements.txt",
    "aquaculture_model.pkl",
    "feature_columns.pkl",
    ".github/workflows/ci.yml"
]

for file in files_to_check:
    status = "✅ EXISTS" if os.path.exists(file) else "❌ MISSING"
    print(f"{file}: {status}")

main.py: ✅ EXISTS
test_api.py: ✅ EXISTS
requirements.txt: ✅ EXISTS
aquaculture_model.pkl: ✅ EXISTS
feature_columns.pkl: ✅ EXISTS
.github/workflows/ci.yml: ✅ EXISTS


In [23]:
import zipfile
import os

project_files = [
    "main.py",
    "test_api.py",
    "requirements.txt",
    "aquaculture_model.pkl",
    "feature_columns.pkl",
    ".github/workflows/ci.yml"
]

with zipfile.ZipFile("aquaculture_cicd_project.zip", "w", zipfile.ZIP_DEFLATED) as zipf:
    for file in project_files:
        zipf.write(file)

print("Project packaged successfully!")
print("ZIP file size:", round(os.path.getsize("aquaculture_cicd_project.zip") / 1024, 2), "KB")

Project packaged successfully!
ZIP file size: 136.09 KB


In [24]:
from google.colab import files

files.download("aquaculture_cicd_project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>